# 03 — Guided construction of the Boudreau-like three-box model

**Learning goal:** translate the full diagram into reservoirs, transported
species, internal connections and boundary fluxes; verify the constructed graph.

**Core time: 55 minutes.** Diagram/workbook (10), four mapping tasks (30),
graph and stationarity checks (10), explanation (5). Native ESBMTK constructors
remain visible. Repeated loops, chemistry/sediment wiring and plotting are supplied.
The notebook does not use `initialize_model` in the student construction path.
[Teaching goals](../../TEACHING_GOALS.md).

## 1. Read the diagram before coding

```text
                      Atmosphere (CO2 mole fraction)
                         ↕ CO2          ↕ CO2
Weathering → Low-latitude surface ─THC→ High-latitude surface
             DIC, TA   ↑                     │  ↕ mixing
                       │ THC                 │ THC
                       └──────── Deep ocean ←┘
                                  DIC, TA

Low-latitude surface ─POC→ Deep ocean          (DIC only)
Low-latitude surface ─PIC→ carbonate module   (1 DIC : 2 TA)
carbonate module ─dissolution→ Deep ocean     (1 DIC : 2 TA)
carbonate module ─burial→ outside active atmosphere–ocean system
```

The carbonate module represents sediment processes and a snowline state; it is
not an explicit sediment-carbon reservoir. Trace the three THC legs, both
mixing directions and both gas connections. The Transport table shows each
water arrow separately. Every fixed-volume box must gain and lose equal water.

**Predict:** which arrows carry both DIC and TA? Which carry carbon only?
Where can the combined atmosphere–ocean carbon inventory change?

## 2. Translate symbols into ESBMTK objects

| Diagram element | Mathematical role | ESBMTK representation |
| --- | --- | --- |
| Simulation boundary and clock | common units and integration interval | `Model` |
| Three ocean boxes | DIC and TA inventories with geometry and water properties | `initialize_reservoirs` |
| Atmosphere | atmospheric carbon inventory expressed as pCO₂ | `GasReservoir` |
| Circulation and mixing | water transport multiplied by source concentration | `create_bulk_connections`, type `scale_with_concentration` |
| Soft-tissue pump | fixed transfer of DIC from low-latitude surface to depth | `POM` connection |
| Carbonate pump | paired surface losses of DIC and TA | `PIC_DIC` and `PIC_TA` connections |
| Carbonate equilibria | DIC and TA determine CO₂(aq), pH, and carbonate ion | carbonate system 1 |
| Carbonate compensation | rain, dissolution, burial, and depth diagnostics | carbonate system 2 |
| Air–sea CO₂ exchange | bidirectional atmosphere–surface carbon flux | `Species2Species`, type `gasexchange` |
| Weathering | fixed external DIC and TA supply | `ConnectionProperties` |

The order matters. Reservoirs must exist before connections can refer to them; surface carbonate chemistry must exist before gas exchange can use aqueous CO₂; and the PIC flux object must exist before carbonate system 2 can consume it.

## 3. Write the conservation equations first

For ocean box $i$ with water mass $m_i$, the two prognostic inventories obey

$$
m_i\frac{d[\mathrm{DIC}]_i}{dt}
=\sum J_{C,\mathrm{in}}-\sum J_{C,\mathrm{out}},
$$

$$
m_i\frac{d[\mathrm{TA}]_i}{dt}
=\sum J_{\mathrm{TA},\mathrm{in}}-\sum J_{\mathrm{TA},\mathrm{out}}.
$$

A transport coefficient $q_{i\rightarrow j}$ in kg/yr carries constituent $X$ at the source concentration (mol/kg),

$$
J^{(X)}_{i\rightarrow j}(t)=q_{i\rightarrow j}[X]_i(t).
$$

The soft-tissue closure exports carbon without TA. Calcification removes one mole of DIC and two equivalents of TA for every mole of CaCO₃,

$$
J_{\mathrm{PIC,TA}}=2J_{\mathrm{PIC,DIC}}.
$$

Dissolution reverses that stoichiometry. Consequently, carbonate export must be represented by a linked DIC–TA pair, never by two independently chosen rates.

**Benchmark unit convention.** For a physical water transport $Q$ in m3/yr,
$q=Q\rho$. The archived ESBMTK benchmark instead passes volume scales through
the native mapper, which converts them to litres/yr; that numerical coefficient
multiplies the mol/kg state. We retain this historical nominal conversion for
benchmark reproduction. The ODE reservoir masses still use each box's ESBMTK
density. Practical 02 explicitly supplies $Q\rho$ in kg/yr. Changing the
benchmark transport convention would be a separate sensitivity experiment.


**Boundary check:** explain why the closed-system inventory test in 02 must
now include weathering and net burial. Chemistry calculates speciation; it
does not generate carbon or TA. Use the supplied audits after construction.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name in {'instructor', 'student'}:
    ROOT = ROOT.parents[1]
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from esbmtk import (
    Q_, ConnectionProperties, GasReservoir, Model, Species2Species,
    add_carbonate_system_1, add_carbonate_system_2,
    create_bulk_connections, initialize_reservoirs,
)
from model import postprocess_carbonate_horizons, run_model, standard_diagnostics
from presets import load_boudreau_parameters
from reservoir_inputs import reservoir_inventory_rows
from model_inputs import read_model_tables

WORKBOOK = ROOT / 'data' / 'Boudreau_2010' / 'model_definition.xlsx'
P = load_boudreau_parameters(WORKBOOK)
input_tables = read_model_tables(WORKBOOK)
STATE = ROOT / 'data' / 'Boudreau_2010' / 'steady_state'

## 4. Read the supplied model definition (10 minutes with the diagram)

The [workbook](../../data/Boudreau_2010/model_definition.xlsx) supplies explicit
areas, volumes, box-specific T/S/P and initial concentrations; atmospheric size
has its own units. Its Transport and GasExchange sheets specify directed arrows.
The notebook displays the tables, so opening Excel is optional.

Keep the benchmark inputs for this practical. The loader validates units,
IDs and per-box water balance without constructing any reservoirs. It supplies
the Python records below; you map their scientific fields into ESBMTK.
PIC is linked to POC by the rain ratio and weathering TA is linked 2:1 to DIC.
Parameter values are owned by the workbook, not duplicated in notebook code.

The archived restart later replaces initial concentrations; it supplies state,
not connections. Changed geometry, thermodynamics, transport or process rates
requires a new stationary restart before interpreting perturbations.

In [ ]:
display(pd.DataFrame(input_tables['OceanReservoirs']).set_index('Box ID'))
display(pd.DataFrame(input_tables['Atmosphere']).set_index('Box ID'))
display(pd.DataFrame(input_tables['BoundaryNodes']).set_index('Box ID'))
display(pd.DataFrame(input_tables['ProcessParameters']).set_index('Parameter'))
print('Derived PIC:', P['pic_export'], '; weathering TA:', P['weathering_ta'])

In [ ]:
# The loader attaches ESBMTK units but creates no reservoirs.
display(pd.DataFrame(P['boxes']).T)

In [ ]:
# Follow one spreadsheet value into the Python record.
print('High-latitude initial DIC:', P['boxes']['H_b']['dic'])

## 5. Supplied simulation container

`Model` defines time, units and chemistry settings; it is not a physical box.
Run this cell unchanged. The four exercises below concern boxes and arrows.

In [ ]:
# Supplied model clock and units.

M = Model(
    stop='20 yr',
    max_timestep='1 yr',
    element=['Carbon', 'Boron', 'Hydrogen', 'misc_variables'],
    mass_unit='mol',
    concentration_unit='mol/kg',
    opt_k_carbonic=P['opt_k_carbonic'],
    opt_pH_scale=P['opt_pH_scale'],
)

assert M.stop == Q_('20 yr').to(M.t_unit).magnitude
assert M.DIC.name == 'DIC' and M.TA.name == 'TA'

## 6. Exercise 03.1: map reservoir fields

Read the supplied high-latitude example. In `ocean_specification`, assign
`concentrations` (keys `M.DIC`, `M.TA`) and `geometry` (keys `area`, `volume`)
from the corresponding `box` record. The loop and T/S/P mapping are supplied.
Explain why atmospheric mole fraction and ocean mol/kg need different inventory
conversions. Boundary nodes have no water volume.

In [ ]:
# Supplied example: Excel row -> Python record -> ESBMTK specification.
high = P['boxes']['H_b']
high_specification = {
    'c': {M.DIC: high['dic'], M.TA: high['ta']},
    'g': {'area': high['area'], 'volume': high['volume']},
    'T': high['temperature'],
    'P': high['pressure'],
    'S': high['salinity'],
}
high_specification

In [ ]:
def ocean_specification(box):
    # Exercise 03.1: complete the two mappings, following high_specification.
    raise NotImplementedError("Exercise: replace this line with your solution")
    return {'c': concentrations, 'g': geometry,
            'T': box['temperature'], 'P': box['pressure'], 'S': box['salinity']}

# Supplied repetition and boundary nodes.
box_parameters = {name: ocean_specification(box) for name, box in P['boxes'].items()}
for node in P['boundary_nodes']:
    box_parameters[node['name']] = {
        'ty': node['type'], 'sp': [getattr(M, species) for species in node['species']],
    }
species_list = initialize_reservoirs(M, box_parameters)
assert {M.L_b.name, M.H_b.name, M.D_b.name} == {'L_b', 'H_b', 'D_b'}
assert set(species_list) == {M.DIC, M.TA}
for box in (M.L_b, M.H_b, M.D_b):
    assert hasattr(box, 'DIC') and hasattr(box, 'TA')

### Check the implemented reservoir inventories

ESBMTK calculates density from each box’s temperature, salinity and pressure. Water mass is volume times this density. The table below shows the resulting water mass and initial carbon/TA amounts. These are software and inventory checks, not extra fitted inputs.

In [ ]:
display(pd.DataFrame(reservoir_inventory_rows(M)).set_index('Box'))

## 7. Exercise 03.2: map water-transport arrows

The thermohaline loop is `L_b → H_b → D_b → L_b`, with 25 Sv on every leg. The separate 30 Sv exchange is `H_b ↔ D_b`. Each arrow uses the **source** concentration. Equal transports in opposing or closed paths conserve water and transport both DIC and TA.

Connection names use `source_to_sink@id`. The part after `@` becomes a readable process identifier for later inspection.

The Transport sheet lists both mixing directions explicitly. Parameter names refer to the Parameters sheet, so the common overturning rate is entered once. Trace each row on the diagram and check that every box gains and loses equal water before constructing the arrows. `Order` preserves the construction sequence if you sort Excel rows.

Complete `source_name`, `sink_name` and `transported_species` for each row.
The supplied loop repeats your mapping; the scale and naming syntax are given.
Trace every row on the diagram before running. State the water balance at H_b.

In [ ]:
display(pd.DataFrame(input_tables['TransportConnections']).sort_values('Order'))
connection_parameters = {}
for row in P['transport_connections']:
    # Exercise 03.2: map each row's source/sink and choose the carried species.
    raise NotImplementedError("Exercise: replace this line with your solution")
    name = f"{source_name}_to_{sink_name}@{row['id']}"
    connection_parameters[name] = {
        'ty': 'scale_with_concentration', 'sc': P[row['parameter']],
        'sp': transported_species,
    }
create_bulk_connections(connection_parameters, M)
assert len(M.loc) == 2 * len(P['transport_connections'])
for row in P['transport_connections']:
    connections = [conn for conn in M.loc if conn.id == row['id']
                   and conn.source.parent.name == row['source']
                   and conn.sink.parent.name == row['sink']]
    assert len(connections) == 2

## 8. Exercise 03.3: choose pump species and stoichiometry

This baseline uses fixed exports. The POC closure moves DIC from the productive low-latitude surface to the deep ocean. PIC removes low-latitude DIC and TA; its nominal DIC sink is bypassed later by carbonate system 2, which partitions the rain between dissolution and burial.

The rain ratio is a diagnostic,

$$
r_{\mathrm{rain}}=\frac{J_{\mathrm{PIC}}}{J_{\mathrm{POC}}}
=\frac{60}{200}=0.3.
$$

Use one quantity for the PIC DIC rate and multiply it by two for the TA rate. This keeps the stoichiometry exact.

Assign the POC source/sink box names and species, and the linked PIC DIC/TA
rates. Templates supply the special PIC sink bypass: only the dissolved fraction
returns to deep water after the carbonate module partitions the rain.

In [ ]:
# Supplied benchmark flux quantities.

M.tutorial_params = P
M.OM_export_reference = Q_(P['poc_export'])
M.CaCO3_export_reference = Q_(P['pic_export'])
M.OM_export = M.OM_export_reference
M.CaCO3_export = M.CaCO3_export_reference

# Exercise 03.3: select the POC arrow and link the two PIC rates.
raise NotImplementedError("Exercise: replace this line with your solution")
create_bulk_connections(
    {
        f"{poc_source}_to_{poc_sink}@POM": {
            'sp': poc_species, 'ty': 'Fixed', 'ra': M.OM_export,
        },
        'L_b_to_D_b@PIC_DIC': {
            'sp': M.DIC, 'ty': 'Fixed', 'ra': pic_dic_rate, 'bp': 'sink',
        },
        'L_b_to_D_b@PIC_TA': {
            'sp': M.TA, 'ty': 'Fixed', 'ra': pic_ta_rate, 'bp': 'sink',
        },
    },
    M,
)

M.OM_export_flux = M.flux_summary(filter_by='POM', return_list=True)[0]
M.CaCO3_export_flux = M.flux_summary(filter_by='PIC_DIC', return_list=True)[0]
M.rain_ratio = (
    M.CaCO3_export.to('Tmol/yr').magnitude
    / M.OM_export.to('Tmol/yr').magnitude
)

poc_connections = [c for c in M.loc if c.id == 'POM']
pic_dic_connections = [c for c in M.loc if c.id == 'PIC_DIC']
pic_ta_connections = [c for c in M.loc if c.id == 'PIC_TA']
assert len(poc_connections) == len(pic_dic_connections) == len(pic_ta_connections) == 1
assert pic_ta_connections[0].rate == 2 * pic_dic_connections[0].rate
assert M.rain_ratio == P['rain_ratio']

## 9. Supplied carbonate chemistry and sediment response

Run the wiring below unchanged. Surface DIC and TA determine aqueous CO2, pH
and carbonate ion. The deep carbonate module receives the actual PIC flux object
and partitions rain between dissolution (returning 1 DIC : 2 TA) and net burial.

The saturation horizon marks saturation = 1; the compensation depth describes
survival of modern carbonate rain; the sediment snowline retains memory of past
conditions. For the core practical, explain the transfers and the slower sediment
response. Detailed equations are [optional reference](../../ref/sediment_reference.md).

In [ ]:
# Supplied chemistry and sediment coupling.

add_carbonate_system_1([M.L_b, M.H_b])
add_carbonate_system_2(
    r_sb=[M.L_b],
    r_db=[M.D_b],
    carbonate_export_fluxes=[M.CaCO3_export_flux],
    z0=P['z0'],
    alpha=P['alpha'],
)

assert hasattr(M.L_b, 'CO2aq') and hasattr(M.H_b, 'CO2aq')
assert M.D_b.cs2.function_input_data[0] is M.CaCO3_export_flux

## 10. Exercise 03.4: connect the atmosphere to surface DIC

`GasReservoir` stores atmospheric CO₂ as a mixing ratio. Each surface box exchanges independently with that common atmosphere. The sign of the flux is calculated from disequilibrium; `source` and `sink` define bookkeeping, not a permanently one-way arrow.

A schematic form of the exchange law is

$$
J_{\mathrm{gas}}=kA\left(K_0p\mathrm{CO}_2-[\mathrm{CO_2(aq)}]\right),
$$

where the solubility $K_0$ depends on the local surface temperature and salinity. The model's solubility pump therefore emerges from chemistry, gas exchange, and circulation rather than from one explicit pump flux.

Inside the supplied loop, assign `gas_source`, `gas_sink` and `gas_species`
using the atmosphere, surface box and species identified by the workbook row.
Use `getattr(M, row['atmosphere'])` for the named atmosphere. Why is the ocean
sink DIC even though the exchange law depends on aqueous CO2?

In [ ]:
display(pd.DataFrame(input_tables['GasExchangeConnections']).sort_values('Order'))

# Supplied atmosphere and loop; complete the scientifically meaningful endpoints.

GasReservoir(
    name='CO2_At', species=M.CO2, species_ppm=P['pco2'],
    reservoir_mass=P['atmosphere_moles'],
)

for row in P['gas_exchange_connections']:
    surface_box = getattr(M, row['surface'])
    # Exercise 03.4: identify atmosphere, surface carbon state and gas species.
    raise NotImplementedError("Exercise: replace this line with your solution")
    Species2Species(
        source=gas_source,
        sink=gas_sink,
        species=gas_species,
        piston_velocity=P[row['parameter']],
        ctype='gasexchange',
        id=surface_box.name,
    )

gas_connections = [c for c in M.loc if c.ctype == 'gasexchange']
assert len(gas_connections) == 2
assert {c.sink for c in gas_connections} == {M.L_b.DIC, M.H_b.DIC}

## 11. Supplied boundary connection: weathering

Weathering supplies DIC and TA to the low-latitude surface in a 1:2 ratio,

$$
J_{\mathrm{weathering,TA}}=2J_{\mathrm{weathering,DIC}}.
$$

At long-term steady state, carbonate burial balances this external supply. This explains why a permanent zero-carbonate-pump case cannot reach the original TA steady state while weathering remains active.

**Explain:** which arrow is the external input, and which process removes carbon
and TA from the active ocean? Why must both appear in its inventory audit?

> **Your explanation:** replace this placeholder with your answer.

In [ ]:
# Supplied external weathering arrow.

ConnectionProperties(
    source=M.Fw,
    sink=M.L_b,
    rate={M.DIC: P['weathering_dic'], M.TA: P['weathering_ta']},
    species=[M.DIC, M.TA],
    ctype='fixed',
    id='weathering',
)

weathering = [c for c in M.loc if c.id == 'weathering']
assert len(weathering) == 2

## 12. Audit the object graph before integrating

A model that runs is not necessarily the model in the diagram. Inspect the constructed graph and check counts, sources, sinks, transported species, and the PIC–sediment coupling before trusting a numerical result.

In [ ]:
connection_table = pd.DataFrame([
    {
        'id': c.id,
        'source': c.source.full_name,
        'sink': c.sink.full_name,
        'type': c.ctype,
    }
    for c in M.loc
])

assert set(connection_table['id']) >= {
    'thc', 'mix_up', 'mix_down', 'POM', 'PIC_DIC', 'PIC_TA',
    'L_b', 'H_b', 'weathering',
}
connection_table.sort_values(['id', 'source']).reset_index(drop=True)

## 13. Load the archived preindustrial state and test stationarity

The archived state avoids repeating the one-million-year benchmark spin-up during class. Loading it is not a substitute for construction: the restart only supplies concentrations and inventories; the flux graph you built still determines every tendency.

An unforced 20-year integration should show only very small numerical drift. A large trend usually means an arrow, species, direction, or stoichiometric factor was translated incorrectly.

In [ ]:
M.read_state(directory=str(STATE))
initial = {
    'atmospheric pCO2 (ppm)': M.CO2_At.c[0] * 1e6,
    'low-latitude DIC (umol/kg)': M.L_b.DIC.c[0] * 1e6,
    'low-latitude TA (umol/kg)': M.L_b.TA.c[0] * 1e6,
}

run_model(M)
postprocess_carbonate_horizons(M)

final = {
    'atmospheric pCO2 (ppm)': M.CO2_At.c[-1] * 1e6,
    'low-latitude DIC (umol/kg)': M.L_b.DIC.c[-1] * 1e6,
    'low-latitude TA (umol/kg)': M.L_b.TA.c[-1] * 1e6,
}

pd.DataFrame({
    'initial': initial,
    'final': final,
    'change': {key: final[key] - initial[key] for key in initial},
})

### 13.1 Verify the reconstructed sediment budget

The post-processing step has now attached all five sediment diagnostics to `M.D_b`. Check both the ordering of the horizons and the carbonate-rain budget at every saved time point.

In [ ]:
sediment_table = pd.DataFrame({
    'zsat (m)': M.D_b.zsat.c,
    'zcc (m)': M.D_b.zcc.c,
    'zsnow (m)': M.D_b.zsnow.c,
    'PIC export (Tmol/yr)': M.D_b.CaCO3_export.c / 1e12,
    'dissolution (Tmol/yr)': M.D_b.Fdiss.c / 1e12,
    'burial (Tmol/yr)': M.D_b.Fburial.c / 1e12,
})

budget_residual = (
    M.D_b.CaCO3_export.c - M.D_b.Fdiss.c - M.D_b.Fburial.c
)
assert abs(budget_residual).max() < 1e-6
assert (M.D_b.zsat.c <= M.D_b.zcc.c).all()

sediment_table.iloc[[0, -1]]

## 14. Trace outputs back to states and arrows

Read this small endpoint table. Identify a prognostic concentration, a chemistry
diagnostic and a flux. The complete benchmark plots are supplied in core 04.

In [ ]:
display(pd.DataFrame({
    b.name: {'DIC (umol/kg)': b.DIC.c[-1] * 1e6,
             'TA (umol/kg)': b.TA.c[-1] * 1e6, 'pH': b.pH.c[-1]}
    for b in (M.L_b, M.H_b, M.D_b)
}).T)

## 16. What you should now be able to explain

1. A flux diagram specifies reservoirs, state variables, internal connections, and boundary fluxes.
2. Physical water transports carry both DIC and TA at source concentrations.
3. The soft-tissue closure carries DIC but no TA.
4. The carbonate pump requires an exact 1:2 DIC–TA pair and feeds carbonate compensation.
5. The solubility pump emerges from temperature-dependent chemistry, gas exchange, and circulation.
6. A restart supplies state, not structure; the model graph must still be correct.

Core 04 now reuses this verified construction to compare carbon and alkalinity additions. Tagged storage and state-dependent feedback hypotheses are separate optional work.

**Finish 03:** point to one reservoir mapping, one water arrow, the POC arrow,
the linked PIC pair, and a gas arrow in your code. Explain why a passing restart
test checks your construction but does not independently validate the model.